# 01. The GIL and free-threading

**Tier:** Deep. **Why you'd use this:** Threaded `x += 1` can lose updates, CPU-bound threads behave differently from waiting threads, and a free-threaded build changes the trade-offs.

**After this lesson you can:** explain that `x += 1` can lose updates; tell CPU-bound threads from threads that wait; say what a free-threaded build changes.
About 30 minutes. **Uses:** [Threads](../../course/17-concurrency/01-threads-lesson.ipynb), [Processes](../../course/17-concurrency/02-processes-lesson.ipynb), [Source to bytecode](../10-bytecode/01-bytecode-lesson.ipynb)


## Quick review

1. When does an object's memory become reusable under reference counting?
2. Why prefer `with` over `__del__` for closing a resource?

<details><summary>Answers</summary>

When the refcount hits zero (cycles need the cyclic GC). `__del__` timing is unreliable; `with` runs `__exit__` at a known point.
</details>


## The idea

On the usual CPython build, one process runs Python bytecode on one thread at a time. The lock that enforces that is the global interpreter lock, the GIL. It protects the interpreter's own data structures, the reference counts and the object headers. It does not protect your variables. `value = value + 1` is several bytecode instructions. The lock can be released between them. Two threads can each read the old value and each write it back.

A free-threaded build can run Python bytecode on many threads at once. The lost update does not go away. It gets easier to hit. Waiting on I/O, including `time.sleep`, releases the GIL even on the usual build, so those threads overlap.

<details><summary>Explain more</summary>

`sys._is_gil_enabled` reports whether this process is holding itself to one bytecode thread. A free-threaded build can report false. Treat that as a CPython detail of the build you launched, not as a property of your program.

`concurrent.interpreters` is a separate isolation tool: more than one interpreter in the process, each with its own GIL state. It is not a way to make `+=` atomic.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

The GIL is not the synchronized keyword and not a memory model. It is a process-wide lock around bytecode. Data races on your own objects are still your lock, your queue, or a process boundary.
</details>


## See it work

A counter that several threads increment has to hold a lock across the read and the write. The lock is a `threading.Lock`. The `with` block is one owner at a time. Two increments then land as `2`.


In [1]:
import threading

value = 0
lock = threading.Lock()


def locked_bump() -> None:
    global value
    # 1. No other thread runs this read and write together.
    with lock:
        value = value + 1


workers = [threading.Thread(target=locked_bump) for _ in range(2)]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()
print(value)


2


| thread | holds the lock | `value` after its write |
| --- | --- | --- |
| first to acquire | yes | 1 |
| second, after the first releases | yes | 2 |

```
thread A: lock, read 0, write 1, unlock
thread B:        waits, then read 1, write 2
```

Without the lock, the predict cell is the schedule you can get. With the lock, that schedule cannot happen.


## Check yourself

Answer these from the example above before you run the cell.


1. Why can the GIL switch threads in the middle of `value = value + 1`?
2. Why do three `sleep` calls overlap when three pure Python loops do not, on a build where the GIL is enabled?
3. What does a free-threaded build not fix?

<details><summary>Answers</summary>

1. The statement is several bytecode instructions. The interpreter can release the GIL between instructions. The load and the store are not one critical section unless you add a lock.
2. `sleep` releases the GIL for the wait, so other threads run. A pure Python loop holds the GIL while it runs bytecode, so the other threads wait their turn.
3. Atomicity of your updates. Several threads can now run the load and the store truly at the same time. You still need a lock, a queue, or no shared mutable state.
</details>


In [2]:
import threading

value = 0
gate = threading.Barrier(2)


def lose_one() -> None:
    global value
    seen = value
    gate.wait()
    value = seen + 1


workers = [threading.Thread(target=lose_one) for _ in range(2)]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()
print(value)


1


<details><summary>Check your answers</summary>

- The print is `1`, not `2`. Both threads copied `0` into `seen` before either wrote. Each then stored `1`. The barrier makes that order happen on every run. The GIL allowed it, because the read and the write are separate steps.
</details>


## Why it works this way

The GIL is a mutex around running bytecode in one interpreter. A thread that wants to execute a Python instruction acquires it. A thread that already holds it drops it every so often, at a check the interpreter inserts, and when the thread would block in I/O or `time.sleep`. Another thread can then run. The switch is between instructions, not in the middle of one C operation marked safe. `value = value + 1` is a load, an add, and a store — separate instructions. The switch can sit between the load and the store. Both threads load the same integer, both add one, both store. One update disappears. The barrier in the predict cell forces both loads first, so the lost update is the result every time.

That is why "the GIL makes Python thread-safe" is wrong. It makes the interpreter's internals safe enough that two threads do not corrupt a reference count. It does not make your `+=` one step. A `Lock` around the whole update makes the update one step. A queue from `queue` is the tool when threads should hand objects across without sharing a counter. A process is the tool when you need a separate memory space.

CPU-bound Python does not speed up with more threads on a build where the GIL stays on — only one thread runs bytecode at a time. I/O-bound threads do overlap, because the waiting thread has released the GIL. The sleep cell is volatile: the claim is that several sleeps finish in about one sleep, not the sum. A process can run bytecode on another core because it has its own interpreter and GIL. Starting a process pickles the target and arguments and pays for a new address space. The timing cell is volatile for the same reason — read it as a measurement including startup.

A free-threaded build lets several threads run bytecode together. `sys._is_gil_enabled` can then be false. CPython detail: the usual build still reports true. Free-threading does not make `value = value + 1` atomic. It removes the accidental serialization that made some races rare. Locks matter more on that build, not less.

`concurrent.interpreters` creates interpreters that do not share the module globals of the main interpreter. They communicate by explicit messages, not by mutating one list. That isolation is the point — a different design from threads, which share memory and therefore share the race.

The design reason for the GIL was a simple way to let C extensions and the interpreter touch objects without a lock on every reference-count update. The cost is no parallel bytecode. Free-threading pays for those locks so bytecode can run in parallel. Your data races were never part of the bargain.


In [3]:
import sys

enabled = getattr(sys, "_is_gil_enabled", None)
print(enabled() if enabled is not None else "absent")


True


In [4]:
import time


def pause() -> None:
    time.sleep(0.15)


started = time.perf_counter()
workers = [threading.Thread(target=pause) for _ in range(3)]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()
print(round(time.perf_counter() - started, 3))


0.156


In [5]:
import importlib.util

available = importlib.util.find_spec("concurrent.interpreters") is not None
print(available)


True


## When to use it

Use threads when the work waits: network, disk, `sleep`, a subprocess you are only watching. Use processes when the work is Python bytecode that fills a core and you have measured that one core is not enough. Use a `Lock` or a `queue` whenever threads share a mutable object. Use `concurrent.interpreters` when you want separate interpreters in one process and can pass messages instead of sharing a list.

Do not add threads to speed up a pure Python loop on a build where the GIL is enabled. Do not remove locks because you read that the GIL exists. A free-threaded build is a reason to be stricter about shared state. The barrier is only for the demonstration — production code holds the lock.


## Common mistake

Two threads each add one to a shared integer with the read and the write separated. The barrier forces both reads to see `0`. The print is `1`. Nothing raises. A test that only checks the type of the result stays green.


In [6]:
value = 0
gate = threading.Barrier(2)


def lose_update() -> None:
    global value
    seen = value
    gate.wait()
    value = seen + 1


workers = [threading.Thread(target=lose_update) for _ in range(2)]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()
print(value)


1


The print is `1`. Both stores wrote the same sum. The fix holds a lock from the read through the write. The second thread cannot read until the first has stored. Two increments produce `2`. The barrier is gone because the lock is what orders the steps.


In [7]:
value = 0
lock = threading.Lock()


def locked_again() -> None:
    global value
    with lock:
        value = value + 1


workers = [threading.Thread(target=locked_again) for _ in range(2)]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()
print(value)


2


## Try it

Change the locked `bump` so it adds `1` a thousand times inside one `with lock` block, and start two threads. Why is the result `2000` even though each thread loops, and what would change if the `with` were inside the loop instead of around it? The result can stay `2000` either way. Which version lets the other thread run between increments?


## What this explains

- [Threads](../../course/17-concurrency/01-threads-lesson.ipynb): threads share memory. The GIL is why CPU-bound Python threads do not scale on the usual build, and why a lock is still required for a shared counter.
- [Processes](../../course/17-concurrency/02-processes-lesson.ipynb): a process has its own interpreter lock and its own memory. Arguments are pickled because nothing is shared.
- [Source to bytecode](../10-bytecode/01-bytecode-lesson.ipynb): `value = value + 1` is more than one instruction. The race sits between those instructions.


## Recap

- The GIL serializes bytecode in one interpreter. It does not make `+=` atomic.
- Waiting releases the GIL. A free-threaded build runs bytecode in parallel and makes races easier to hit.
- Hold a lock across the whole update, or do not share the object. `concurrent.interpreters` isolates interpreters. It does not fix a shared counter.

Next: the question bank that collects these answers in one place is already growing beside this track.
